# Exploratory Data Analysis — IBM Telco Customer Churn

**Goal:** Understand the data structure, class imbalance, and which features correlate most strongly with churn — before any modelling.

**Dataset:** [IBM Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
7,043 customers · 20 features · binary target (`Churn`)

---
**Table of Contents**
1. Setup & Load
2. Dataset Overview
3. Target Variable — Class Imbalance
4. Numerical Features
5. Categorical Features & Churn Rates
6. Key Business Segments
7. Correlation Analysis
8. Summary of Insights

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded.')

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

RAW_CSV = '../data/raw/Telco-Customer-Churn.csv'

df = pd.read_csv(RAW_CSV)

# Fix TotalCharges — whitespace entries → NaN → median impute
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Binary target
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 1. Dataset Overview

In [ ]:
print('=== SHAPE ===')
print(f'  Rows   : {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')

print('\n=== DTYPES ===')
print(df.dtypes.to_string())

print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.sum() > 0 else 'No missing values after imputation.')

In [ ]:
df.describe()

**Key observations:**
- `tenure` ranges 0–72 months. Mean ~32 months.
- `MonthlyCharges` ranges \$18–\$119. Mean ~\$65.
- `TotalCharges` = tenure × monthly (approx) — highly correlated with tenure.
- 11 rows had `TotalCharges` as whitespace (new customers, tenure=0). Imputed with median.

---
## 2. Target Variable — Class Imbalance

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate   = df['Churn_bin'].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = axes[0].bar(churn_counts.index, churn_counts.values,
                   color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Churn Count', fontweight='bold')
axes[0].set_ylabel('Customers')
axes[0].set_ylim(0, churn_counts.max() * 1.15)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Rate', fontweight='bold')

fig.suptitle(f'Target Distribution — Churn Rate: {churn_rate:.1%}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nChurn rate: {churn_rate:.1%}  ({churn_counts["Yes"]:,} churners / {len(df):,} total)')
print(f'Class ratio (neg:pos): {churn_counts["No"]/churn_counts["Yes"]:.1f}:1')

**~26.5% churn rate** — moderate imbalance (~3.7:1 ratio). Not extreme, but enough that naive accuracy would be misleading (a model predicting "no churn" for everyone gets 73.5% accuracy). We use `scale_pos_weight` in XGBoost to handle this.

---
## 3. Numerical Features

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, numeric_cols):
    for label, color, grp in [('No Churn', '#2ecc71', df[df['Churn']=='No'][col]),
                               ('Churn',    '#e74c3c', df[df['Churn']=='Yes'][col])]:
        ax.hist(grp, bins=30, alpha=0.6, color=color, label=label, density=True)
        ax.axvline(grp.mean(), color=color, linestyle='--', linewidth=1.5)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Numerical Feature Distributions by Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Mean values by churn status:')
print(df.groupby('Churn')[numeric_cols].mean().round(2).to_string())

**Key numerical findings:**
| Feature | No Churn | Churn | Difference |
|---|---|---|---|
| tenure | ~38 months | ~18 months | **Churners leave ~2× sooner** |
| MonthlyCharges | ~$61 | ~$74 | **Churners pay $13/mo more** |
| TotalCharges | ~$2,556 | ~$1,532 | (driven by shorter tenure) |

The tenure difference is the strongest signal — new customers are at dramatically higher risk.

---
## 4. Categorical Features & Churn Rates

In [ ]:
def churn_rate_plot(col: str, ax, title: str = None):
    """Bar plot of churn rate per category value."""
    rates = df.groupby(col)['Churn_bin'].mean().sort_values(ascending=False)
    colors = ['#e74c3c' if r > 0.30 else '#f39c12' if r > 0.15 else '#2ecc71'
              for r in rates.values]
    bars = ax.bar(rates.index, rates.values * 100, color=colors, edgecolor='white')
    ax.axhline(churn_rate * 100, color='navy', linestyle='--', linewidth=1.2,
               label=f'Overall {churn_rate:.1%}')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_title(title or col, fontweight='bold')
    ax.set_ylim(0, min(rates.max() * 120, 100))
    for bar, val in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.0%}', ha='center', fontsize=9)
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=20)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
churn_rate_plot('Contract',         axes[0,0], 'Contract Type')
churn_rate_plot('InternetService',  axes[0,1], 'Internet Service')
churn_rate_plot('TechSupport',      axes[0,2], 'Tech Support')
churn_rate_plot('OnlineSecurity',   axes[1,0], 'Online Security')
churn_rate_plot('PaymentMethod',    axes[1,1], 'Payment Method')
churn_rate_plot('PaperlessBilling', axes[1,2], 'Paperless Billing')

fig.suptitle('Churn Rate by Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
churn_rate_plot('MultipleLines',   axes[0], 'Multiple Lines')
churn_rate_plot('gender',          axes[1], 'Gender')
churn_rate_plot('SeniorCitizen',   axes[2], 'Senior Citizen')
plt.tight_layout()
plt.show()

**Categorical findings — ranked by signal strength:**

| Feature | High-churn group | Churn rate | Low-churn group | Churn rate |
|---|---|---|---|---|
| **Contract** | Month-to-month | ~43% | Two year | ~3% |
| **Internet** | Fiber optic | ~42% | No internet | ~7% |
| **Tech Support** | No support | ~42% | Has support | ~15% |
| **Online Security** | No security | ~42% | Has security | ~15% |
| **Payment** | Electronic check | ~45% | Credit card | ~15% |
| **Paperless** | Yes | ~34% | No | ~16% |
| **Gender** | (no difference) | ~26% | (no difference) | ~26% |

**Gender is not predictive** — equal churn rates. Senior Citizen shows modest signal.

---
## 5. Key Business Segments

In [ ]:
# Tenure segments
df['tenure_segment'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0–12 mo (new)', '13–24 mo', '25–48 mo', '49–72 mo (loyal)'],
    right=True
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by tenure segment
seg_rates = df.groupby('tenure_segment', observed=True)['Churn_bin'].mean() * 100
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
bars = axes[0].bar(seg_rates.index, seg_rates.values, color=colors, edgecolor='white')
axes[0].axhline(churn_rate*100, color='navy', linestyle='--', linewidth=1.2, label=f'Overall {churn_rate:.1%}')
for bar, val in zip(bars, seg_rates.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.0f}%', ha='center', fontweight='bold')
axes[0].set_title('Churn Rate by Tenure Segment', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

# MonthlyCharges vs Churn — violin plot
sns.violinplot(data=df, x='Churn', y='MonthlyCharges',
               palette={'No': '#2ecc71', 'Yes': '#e74c3c'},
               inner='quartile', ax=axes[1])
axes[1].set_title('Monthly Charges Distribution by Churn', fontweight='bold')
axes[1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

print('Customer counts and churn rates by tenure segment:')
summary = df.groupby('tenure_segment', observed=True).agg(
    n_customers=('Churn_bin', 'count'),
    churn_rate=('Churn_bin', 'mean'),
    avg_monthly=('MonthlyCharges', 'mean')
).round(3)
print(summary.to_string())

In [ ]:
# The dangerous combo: Month-to-month + Fiber optic
pivot = df.groupby(['Contract', 'InternetService'])['Churn_bin'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) — Contract × Internet Service', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nHigh-risk segment: Month-to-month + Fiber optic')
hr = df[(df['Contract']=='Month-to-month') & (df['InternetService']=='Fiber optic')]
print(f'  Count: {len(hr):,} customers ({len(hr)/len(df):.1%} of total)')
print(f'  Churn rate: {hr["Churn_bin"].mean():.1%}')
print(f'  Avg monthly charges: ${hr["MonthlyCharges"].mean():.0f}')

**Business insight — highest risk segment:**  
Month-to-month + Fiber optic customers churn at **~53%**. They make up ~22% of the customer base but generate a disproportionate share of churn. These should be the primary target for retention campaigns.

---
## 6. Correlation Analysis

In [ ]:
# Encode categoricals for correlation matrix
df_enc = df.copy()

binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_enc[col] = df_enc[col].map(binary_map)

# Internet-related services: map to 0/1 (No internet service → 0)
inet_map = {'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0}
for col in ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
    df_enc[col] = df_enc[col].map(inet_map)

# Contract: ordinal
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df_enc['Contract'] = df_enc['Contract'].map(contract_map)

# Internet service
internet_map = {'No': 0, 'DSL': 1, 'Fiber optic': 2}
df_enc['InternetService'] = df_enc['InternetService'].map(internet_map)

# Drop unused
df_enc = df_enc.drop(columns=['customerID', 'Churn', 'tenure_segment', 'PaymentMethod'])

corr = df_enc.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target — ranked
target_corr = corr['Churn_bin'].drop('Churn_bin').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr.values]
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Churn (Pearson r)', fontweight='bold')
ax.set_xlabel('Correlation coefficient')
for bar, val in zip(bars, target_corr.values):
    x = val + 0.005 if val >= 0 else val - 0.005
    ax.text(x, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print('Top 5 positive correlators (increase churn risk):')
print(target_corr[target_corr > 0].head().to_string())
print('\nTop 5 negative correlators (decrease churn risk):')
print(target_corr[target_corr < 0].head().to_string())

**Correlation findings:**
- Strongest positive (↑ churn): `MonthlyCharges`, `InternetService` (Fiber), `PaperlessBilling`, `SeniorCitizen`
- Strongest negative (↓ churn): `tenure`, `Contract` (longer = safer), `TechSupport`, `OnlineSecurity`
- `gender` has correlation ≈ 0 — confirms it's not predictive
- `TotalCharges` and `tenure` are highly correlated (~0.82) — multicollinearity, but tree models handle this well

---
## 7. Summary of Insights

| Rank | Insight | Business Action |
|---|---|---|
| 1 | **Month-to-month contracts → 43% churn** vs 3% for 2-year | Incentivize contract upgrades |
| 2 | **Fiber optic → 42% churn** — likely price dissatisfaction | Review fiber pricing / add value |
| 3 | **New customers (tenure < 12 mo) → highest churn** | Onboarding program + early check-ins |
| 4 | **No Tech Support / Online Security → 42% churn** | Bundle security + support at signup |
| 5 | **Electronic check payment → 45% churn** | Nudge to automatic payment methods |
| 6 | **Gender has zero predictive power** | Drop from model or leave for fairness |
| 7 | **High monthly charges without long contract** | Highest-risk combo; prioritise in retention |

**Modelling implication:** Contract type, tenure, and internet service are the top features — consistent with SHAP analysis on the trained XGBoost model.

In [ ]:
# Revenue at risk from churners
AVG_CLV_MONTHS = 12
churners = df[df['Churn'] == 'Yes']
total_clv_at_risk = (churners['MonthlyCharges'] * AVG_CLV_MONTHS).sum()

print('=== REVENUE AT RISK (full dataset) ===')
print(f'  Total churners         : {len(churners):,}')
print(f'  Avg monthly charges    : ${churners["MonthlyCharges"].mean():.2f}')
print(f'  Total CLV at risk (12mo): ${total_clv_at_risk:,.0f}')
print(f'  Avg CLV per churner    : ${churners["MonthlyCharges"].mean() * AVG_CLV_MONTHS:,.0f}')
print()
print('→ A 10% improvement in retention rate would save approximately')
print(f'  ${total_clv_at_risk * 0.10:,.0f} in annual revenue.')